# P4-M1 CPU development
Use the existing Kaggle notebook with Internet and GITHUB_TOKEN. Save & Run All executes the frozen control and single Cell study and returns the manifest and report. Attach saved Output to retry publication.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile

remote = 'https://github.com/ArcheLabs/mini-cells.git'
repo = Path('/kaggle/working/minicells-p4')
branch = 'research/p4-peer-titans-m0-m1'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
base = repo / 'research/projects/cellular-language-models/experiments/p4-peer-titans-collaboration'
out = Path('/kaggle/working/p4-m1-' + source[:12])
manifest_path = out / 'run-manifest.json'
if not manifest_path.exists():
    for candidate in Path('/kaggle/input').rglob(out.name + '/run-manifest.json'):
        previous = json.loads(candidate.read_text())
        if previous.get('status') == 'M1_COMPLETE':
            shutil.copytree(candidate.parent, out)
            break
if not manifest_path.exists():
    subprocess.run([sys.executable, str(base/'run_m1.py'), '--output', str(out)],
                   cwd=repo, check=True, timeout=600)
manifest = json.loads(manifest_path.read_text())
if manifest['status'] != 'M1_COMPLETE':
    raise RuntimeError('P4-M1 gate failed. Inspect retained run-manifest.json.')
for key, path in [('runner_sha256', base/'run_m1.py'), ('config_sha256', base/'configs/m1.json')]:
    if manifest[key] != hashlib.sha256(path.read_bytes()).hexdigest():
        raise RuntimeError('P4-M1 output identity mismatch.')
manifest['commit_sha'] = source
report = subprocess.check_output([sys.executable, str(base/'report.py'), str(manifest_path)],
                                 text=True, timeout=60)
(out/'REPORT.md').write_text(report)
sys.path.insert(0, str(repo))
from scripts.research.publish_experiment_results import ExperimentSpec, prepare_artifacts, push_results

run_id = out.name
result_branch = 'kaggle/' + run_id
with tempfile.TemporaryDirectory(prefix='p4-publish-') as tmp:
    root = Path(tmp)/'repo'
    subprocess.run(['git', 'clone', '--quiet', '--shared', str(repo), str(root)], check=True, timeout=180)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote], cwd=root, check=True)
    stage = root/'results/p4-m1'
    stage.mkdir(parents=True)
    (stage/'run-manifest.json').write_text(json.dumps(manifest, indent=2))
    (stage/'REPORT.md').write_text(report)
    spec = ExperimentSpec('results/p4-m1', 'artifacts/experiments/p4-peer-titans/'+run_id,
                          ('run-manifest.json', 'REPORT.md'), result_branch, 'minicells.p4-m1.v1')
    folder = prepare_artifacts(root, 'p4-peer-titans', spec, None)
    push_results(root, folder, 'p4-peer-titans', result_branch, 'GITHUB_TOKEN', 'origin/main')
    sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
(out/'publication.json').write_text(json.dumps({'status':'REMOTE_SHA_VERIFIED','commit':sha,'branch':result_branch}))
